This code aims to take a shapefile of station locations and measuements and removes a subset of stations at random to rerun in Greg's code. This could be implemented in some sort of potential cross validation.

In [1]:
import sys
import os as os

import geopandas as gpd
import numpy as np
from numpy.polynomial import Polynomial
import psycopg2
from netCDF4 import Dataset

from tqdm import tqdm
from multiprocessing import Pool
import statsmodels.api as sm
from scipy import stats
# from scipy import optimize

import cartopy.crs as ccrs
from cartopy.feature import NaturalEarthFeature as cfNEF

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import patches
#import matplotlib.patches as patches
import matplotlib.patheffects as path_effects
from matplotlib.lines import Line2D

import rasterio
import xarray as xr

import random
import subprocess
import netCDF4
import shutil

import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.model_selection import ShuffleSplit

This is the updated code using shuffle split cross validator to remove test stations:

In [15]:
# Inputs for code:
net_CDF1_Path = "C:/Users/clemasters/Research Triangle Institute/CIROH Basecamp - Documents/Projects/0218723.014 - SNODAS/Data/snodas_assims_2023_2024/snodas_assim_20221024/ssm1054_2022102412.nc" 
net_CDF2_Path = "C:/Users/clemasters/Research Triangle Institute/CIROH Basecamp - Documents/Projects/0218723.014 - SNODAS/Data/snodas_assims_2023_2024/snodas_assim_20221024/ssm_process_region_2022102312_2022102412_swe_us.nc"
net_CDF_name_us = "ssm_process_region_2022102312_2022102412_swe_us.nc"
net_CDF_name = "ssm1054_2022102412.nc"
folder_name = "snodas_assim_20221024"
shapefile_name = "ssm1054_md_based_2022102312_2022102412_us.shp"
start_date = "2022102312" # These are the numbers within the shapefile like shown in shapefile name above
end_date = "2022102412" # These are the numbers within the shapefile like shown in shapefile name above

# Load the shapefile:
shapefile_path = f"C:/Users/clemasters/Research Triangle Institute/CIROH Basecamp - Documents/Projects/0218723.014 - SNODAS/Data/snodas_assims_2023_2024/{folder_name}/{shapefile_name}"
gdf = gpd.read_file(shapefile_path)

# Filter out rows rows with 'SNOTEL' in 'STATION_TY' switch this with UMRB Stations!!!!
gdf_filtered = gdf[gdf['STATION_TY'] == 'SNOTEL']

# Initialize ShuffleSplit
sss = ShuffleSplit(n_splits=1, test_size=0.10, random_state=0) # n_splits define how many test runs to do and test_size takes a percentage of gdf_filtered as test group

# Generate indices for splitting
for i, (train_index, test_index) in enumerate(sss.split(gdf_filtered)):  # Use gdf_filtered for generating test indices
    print(f"Fold {i}:")
    print(f"  Train: index={train_index}")
    print(f"  Test:  index={test_index}")
    
    # Extract training subset from gdf except the rows in test_subset
    train_subset = gdf.drop(test_index)
    
    # Extract test subset from gdf_filtered
    test_subset = gdf_filtered.iloc[test_index]
    
    # Print the size of each subset
    print(f"  Train subset size: {len(train_subset)}")
    print(f"  Test subset size: {len(test_subset)}")
    
    # Change folder and shapefile names
    folder_name = f"snodas_assim_20221024{i}"
    shapefile_name = f"ssm1054_md_based_2022102312_2022102412_us{i}.shp"
    net_CDF_name = f"ssm_process_region_2022102312_2022102412_swe_us{i}.nc"
    
    
    # Write new shapefile to new folder - had to save twice to get correct shapefile name - Ideas???
    train_subset.to_file(f"C:/Repos/umrb-snodas/Test_Runs/{folder_name}")
    train_subset.to_file(f"C:/Repos/umrb-snodas/Test_Runs/{folder_name}/{shapefile_name}")
    

    
    # Copy netCDF's from orgignal folder to new folder
    shutil.copy(net_CDF1_Path,f"C:/Repos/umrb-snodas/Test_Runs/{folder_name}/{net_CDF_name}" )
    shutil.copy(net_CDF2_Path,f"C:/Repos/umrb-snodas/Test_Runs/{folder_name}/{net_CDF_name_us}" )
    
    # Enter command you want to run    
    cmd = f"python ./snodas_idw.py -k -g -i 3 50 1 0 250 {start_date} {end_date} us{i} -f ../Test_Runs/{folder_name}" # Input the command from Greg's examples
    
    # Run Greg's code using the above inputs and chosen command in cmd line
    subprocess.run(cmd, capture_output=True)

Fold 0:
  Train: index=[ 56 187 136 182 191 152  63  60 161  33   4  55  96  44  45  26 178 180
 113   8 101  89  90 118 175 111  24  30 112  61 177 123  19 146  83  54
 156  16  51  86 143  40 176  22 131 130 137  80 108  14  27  92 109  46
 188  98  62   2  59 106 107  43  10  93  73 185 171 124 189 134 154 184
  50   0  94 153  95  64 135  41  69  49  48  85  13 144  23 179 129  20
  15  78 104  52 100  76   3 116 190 125   6  68  75  84 141 121  12 168
 164 149  91 173  11 119 102  35  57  65   1 120 155  42 105 132 166  17
  38 133  53 150 128  34  28 114 151  31 159 127 169  32 142 162 147  29
  99  82  79 115 148 186  72  77  25 165  81 181 174 183  39  58 140  88
  70  87  36  21   9 103  67 117  47 172]
  Test:  index=[110  74 163  97 126  71  18 157 145   7   5 139 158 170 160 167  37  66
 138 122]
  Train subset size: 2260
  Test subset size: 20


Notes for Paul:

The above code uses a shuffle split cross validator to remove a test group of stations from the shapefile. It is able to take any assimilation file that we downloaded and with updataing the input field should be able to run a new assimilation on the updated shapefile with the test stations removed. A new net_CDF output file and nudging image for the run is saved in the source folder within the Repo. The newly created shapefiles that are created for this run are saved in their own folders within UMRB-SNODAS Repo called Test_Runs. 


In [ ]:
# Load the shapefile
shapefile_path = "C:\\Repos\\umrb-snodas\\examples\\snodas_assim_20221112\\ssm1054_md_based_2022111112_2022111212_us.shp" 
gdf = gpd.read_file(shapefile_path)


# Filter out rows where latitude or longitude is NaN and keep only rows with 'SNOTEL' in 'STATION_TY'
gdf_filtered = gdf[gdf['STATION_TY'] == 'SNOTEL']

# List to store subsets
subsets_list = []

for x in range(1):
    
    gdf = gpd.read_file(shapefile_path)
    
    # Randomly select a subset of stations
    subset_size = 10
    if len(gdf_filtered) < subset_size:
        print("Error: Subset size is larger than the number of stations.")
    else:
        valid_indices = gdf_filtered.index.tolist()
        subset_indices = random.sample(valid_indices, subset_size)
        subset = gdf_filtered.loc[subset_indices]  # Use gdf_filtered instead of gdf

        # Append the subset DataFrame to the list
        subsets_list.append(subset)
        
        # Drop subsets from original gdf
        gdf = gdf.drop(subset_indices)

        # Write new shapefile to new folder - had to save twice to get correct shapefile name - Ideas???
        gdf.to_file(f"C:/Repos/umrb-snodas/examples/snodas_assim_20221112{x}")
        gdf.to_file(f"C:/Repos/umrb-snodas/examples/snodas_assim_20221112{x}/ssm1054_md_based_2022111112_2022111212_us{x}.shp")
        
        # Copy netCDF's from orgignal folder to new folder
        shutil.copy('C:\\Repos\\umrb-snodas\\examples\\snodas_assim_20221112\\ssm1054_2022111212.nc',f"C:/Repos/umrb-snodas/examples/snodas_assim_20221112{x}/ssm1054_2022111212.nc" )
        shutil.copy('C:\\Repos\\umrb-snodas\\examples\\snodas_assim_20221112\\ssm_process_region_2022111112_2022111212_swe_us.nc',f"C:/Repos/umrb-snodas/examples/snodas_assim_20221112{x}/ssm_process_region_2022111112_2022111212_swe_us{x}.nc" )
        
        cmd = f"python ./snodas_idw.py -k -g -i 3 50 1 0 250 2022111112 2022111212 us{x} -f ../examples/snodas_assim_20221112{x}"
   
        subprocess.run(cmd, capture_output=True)
        
        print(subsets_list)